# CYMEK FORMATION-MUX-001 — Kaggle T4 x2

## BEFORE RUNNING

Kaggle **Settings → Accelerator → GPU T4 x2** and **Internet → ON**.

Kaggle currently documents 12-hour CPU/GPU notebook sessions and 20 GB of auto-saved `/kaggle/working` storage. T4 x2 provides two NVIDIA T4 GPUs. This notebook uses the two T4s as independent arm workers (no DDP).

This notebook runs frozen **Science S5**. The campaign may take more than one Kaggle session: it never shortens the scientific exposure to fit a session. Near the per-session wall it stops launching new work, preserves exact-resume state, packages results, and continues from that output in the next session.

S5 uses 60,000 unique training rows (10,000 per family), 480 development rows, and 720 sealed rows. CS-MECH checkpoints every 200 updates. REP-FORM remains scientifically matched by processed tokens and checkpoints every 10,000 processed tokens. Every checkpoint writes `LATEST_PROGRESS.json` plus an immutable `progress/UPDATE_*.json` snapshot. Raw sealed examples are never persisted.

The operator runs an S5-specific compile/test qualification on Kaggle, a dual-T4 engineering-only E2E qualification, calibration, and a storage-capacity preflight before official arms start.

If status is `PARTIAL_SESSION`, save the Kaggle version/output, attach that output to the next run, and rerun this exact notebook.


In [ ]:
# CELL 1 — fetch and verify immutable operator + Science S5
import pathlib, shutil, subprocess, sys
REPO = pathlib.Path('/kaggle/working/An-Ra-the-new-AGI')
REMOTE = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
OPERATOR_COMMIT = 'fdc2483fed0bb1a80d4bfad376aac84a66db6055'
OPERATOR_PATH = 'tools/formation_mux_001_kaggle_operator_v8.py'
OPERATOR_BLOB = '58614bcab3d5a5ea4b3b28e00da56a834c321bba'
SCIENCE_COMMIT_S5 = 'c15ad8beb409537db42d075684ea54847a074ebd'
if REPO.exists() and not (REPO / '.git').exists():
    shutil.rmtree(REPO)
if not REPO.exists():
    subprocess.run(['git', 'clone', REMOTE, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'clean', '-fdx'], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-f', '-q', OPERATOR_COMMIT], check=True)
head = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
assert head == OPERATOR_COMMIT, (head, OPERATOR_COMMIT)
blob = subprocess.run(['git', '-C', str(REPO), 'hash-object', OPERATOR_PATH], capture_output=True, text=True, check=True).stdout.strip()
assert blob == OPERATOR_BLOB, (blob, OPERATOR_BLOB)
subprocess.run(['git', '-C', str(REPO), 'cat-file', '-e', SCIENCE_COMMIT_S5 + '^{commit}'], check=True)
try:
    import tokenizers
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers'], check=True)
    import tokenizers
disk = shutil.disk_usage('/kaggle/working')
print('PYTHON:', sys.version.split()[0], '| TOKENIZERS:', tokenizers.__version__)
print('KAGGLE WORKING GiB total/free:', round(disk.total / 1024**3, 2), '/', round(disk.free / 1024**3, 2))
print('OPERATOR VERIFIED:', OPERATOR_COMMIT)
print('OPERATOR BLOB VERIFIED:', OPERATOR_BLOB)
print('SCIENCE S5 AVAILABLE:', SCIENCE_COMMIT_S5)


In [ ]:
# CELL 2 — one canonical long-run path
import pathlib, subprocess, sys
root = pathlib.Path('/kaggle/working/FORMATION_MUX_001')
code = subprocess.run([
    sys.executable, '-u', 'tools/formation_mux_001_kaggle_operator_v8.py',
    '--repo', '/kaggle/working/An-Ra-the-new-AGI',
    '--out', str(root),
], cwd='/kaggle/working/An-Ra-the-new-AGI')
if code.returncode != 0:
    failure = root / 'GLOBAL_FAILURE.json'
    if failure.exists(): print('GLOBAL_FAILURE:', failure.read_text())
    raise RuntimeError('FORMATION-MUX failed closed with exit ' + str(code.returncode) + '; preserve the Kaggle Output and result ZIP')


In [ ]:
# CELL 3 — status / checkpoint / result retrieval
import json, pathlib
root = pathlib.Path('/kaggle/working/FORMATION_MUX_001')
bundle = pathlib.Path('/kaggle/working/FORMATION_MUX_001_RESULTS.zip')
state = json.loads((root / 'CAMPAIGN_STATE.json').read_text()) if (root / 'CAMPAIGN_STATE.json').exists() else {}
print('STATUS:', state.get('status'))
print('ARMS:', state.get('complete_arms'), '/', state.get('required_arms'))
print('PENDING SEED BUNDLES:', state.get('pending_seed_bundles'))
print('WALL GUARD:', state.get('wall_guard_triggered'))
q = root / 'QUALIFICATION.json'
if q.exists():
    qr = json.loads(q.read_text())
    print('QUALIFICATION:', qr.get('status'))
s = root / 'STORAGE_PREFLIGHT.json'
if s.exists():
    sr = json.loads(s.read_text())
    print('STORAGE PREFLIGHT:', sr.get('pass'), '| free GiB=', round(sr.get('disk_free_bytes', 0)/1024**3, 2), '| required GiB=', round(sr.get('required_free_bytes', 0)/1024**3, 2))
latest = sorted(root.rglob('LATEST_PROGRESS.json'))
print('LATEST PROGRESS SNAPSHOTS:', len(latest))
for p in latest:
    x = json.loads(p.read_text())
    print(p.relative_to(root), 'updates=', x.get('updates'), 'processed=', x.get('processed_tokens'), 'formation=', x.get('formation_so_far'))
for exp in ('CS-MECH-002', 'REP-FORM-003A'):
    p = root / exp / 'FINAL_RESULT.json'
    print(exp, '->', json.loads(p.read_text()).get('verdict') if p.exists() else 'not finalized')
print('RESULT ZIP:', bundle, 'exists=', bundle.exists())
if state.get('status') == 'PARTIAL_SESSION':
    print('NEXT: save this Kaggle version/output, attach that output to the next run, and rerun this exact notebook.')
